In [ ]:
import os
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def merge_lora_to_base_safe():
    # --- 配置区域 ---
    # 必须指向你微调时用的那个基础模型
    base_model_path = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" 
    # 效果最好的 checkpoint 路径
    lora_path = "outputs_yue_qwen/checkpoint-10000" 
    # 合并后完整模型的保存路径
    output_path = "final_merged_model_safe"

    print(f"🚀 步骤 1: 正在加载分词器...")
    tokenizer = AutoTokenizer.from_pretrained(lora_path, trust_remote_code=True)

    print(f"🚀 步骤 2: 正在从 CPU 加载基础模型 (bnb-4bit)...")
    # 注意：我们这里先加载 4bit 模型
    from unsloth import FastLanguageModel
    base_model, _ = FastLanguageModel.from_pretrained(
        model_name = base_model_path,
        max_seq_length = 2048,
        load_in_4bit = True,
        device_map = "cpu", # 强制在 CPU 合并，防止显存 OOM 导致的截断
    )

    print(f"🚀 步骤 3: 正在挂载 LoRA 权重...")
    # 使用 PEFT 原生方式挂载，避免 Unsloth 的加速 Patch 干扰合并
    model = PeftModel.from_pretrained(
        base_model,
        lora_path,
        device_map = "cpu",
        torch_dtype = torch.float16
    )

    print(f"🚀 步骤 4: 核心融合 (Merge & Unload)...")
    # merge_and_unload 会将 LoRA 权重加回到基础权重中
    # Unsloth 对象的 save_pretrained_merged 在这里可能会出问题，所以我们用原生的
    model = model.merge_and_unload()

    print(f"🚀 步骤 5: 正在保存完整 FP16 模型至 {output_path}...")
    # 显式保存为 float16，确保推理兼容性
    model.save_pretrained(
        output_path, 
        safe_serialization=True, 
        max_shard_size="4GB"
    )
    tokenizer.save_pretrained(output_path)

    print(f"✅ 任务完成！")
    print(f"现在你可以使用如下代码测试：")
    print(f"model = AutoModelForCausalLM.from_pretrained('{output_path}', torch_dtype=torch.float16, device_map='auto')")

if __name__ == "__main__":
    # 屏蔽不必要的警告
    import warnings
    warnings.filterwarnings("ignore")
0

In [ ]:
import os
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"
from unsloth import FastLanguageModel
import torch

# 1. 重新加载模型 (一定要 load_in_4bit = False 才能出最高精度的 GGUF)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "outputs_yue_qwen/checkpoint-10000",
    max_seq_length = 2048,
    load_in_4bit = False, # 200GB 空间够了，我们追求最高质量
)

# 2. 一键导出 GGUF (Mac Mini 最佳量化格式)
print("🚀 空间已充裕，正在开始 GGUF 导出（含合并步骤）...")

model.save_pretrained_gguf(
    "./yue_model_final", 
    tokenizer, 
    quantization_method = "q4_k_m", # Mac Mini M系列芯片运行最稳、速度最快的格式
)

print("🎉 如果看到这一行，说明 GGUF 文件已经静静地躺在 /workspace/yue_model_final_gguf 文件夹里了！")